# TIES HPSC-ES Working Group, November 6, 2026: mixed precision computing in PyTorch

Runtime > Change runtime type > Python 3, Hardware accelerator T4 GPU before running. The CPU sections run without a GPU; the GPU sections need one.

## Setup

PyTorch is preinstalled on Colab. A tensor carries its precision as `dtype`, and `.float()`, `.half()` and `.to(dtype)` make a copy in another precision.

In [ ]:
import torch, time

In [ ]:
torch.__version__, torch.version.cuda  # PyTorch and CUDA versions

In [ ]:
torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'

## Cholesky factorization in double and single precision

n random locations, the exponential covariance between them with effective range 0.1, in double and in single. Single takes half the memory, is faster, and gives a factor close to the double one.

In [ ]:
torch.manual_seed(1)
n = 10240
x = torch.rand(n, 2, dtype=torch.float64)  # n random locations

In [ ]:
S64 = torch.exp(-2.9957 * torch.cdist(x, x) / 0.1)  # exponential covariance, effective range 0.1

In [ ]:
S32 = S64.float()  # 32-bit copy

In [ ]:
S64.nbytes / 2**30, S32.nbytes / 2**30  # memory in GB

In [ ]:
t = time.time(); L64 = torch.linalg.cholesky(S64); time.time() - t  # double

In [ ]:
t = time.time(); L32 = torch.linalg.cholesky(S32); time.time() - t  # single

In [ ]:
(L64 - L32).abs().max().item()  # largest entrywise difference

In [ ]:
2 * L64.diagonal().log().sum().item(), 2 * L32.diagonal().log().sum().item()  # log determinants

## Cholesky may break down in low precision

Same locations and effective range, Matérn 5/2 covariance: the matrix is closer to singular, and in single precision the factorization finds a non-positive pivot and stops.

In [ ]:
del S64, S32, L64, L32  # free memory

In [ ]:
h = 5.9154 * torch.cdist(x, x) / 0.1  # same locations, effective range 0.1

In [ ]:
S64 = (1 + h + h**2 / 3) * torch.exp(-h); del h  # Matern 5/2 covariance

In [ ]:
S32 = S64.float()  # 32-bit copy

In [ ]:
t = time.time(); L64 = torch.linalg.cholesky(S64); time.time() - t  # double

In [ ]:
try:
    t = time.time(); L32 = torch.linalg.cholesky(S32); print(time.time() - t)  # single: expect an error
except Exception as e:
    print(type(e).__name__, e)

In [ ]:
2 * L64.diagonal().log().sum().item()  # log determinant, double

## GPU run, exponential covariance

Move the locations to the GPU; everything built from them lives there too. A T4 does single precision much faster than double, so the gap between the two is wider than on the CPU.

In [ ]:
del S64, S32, L64  # free memory

In [ ]:
x = x.cuda()  # move the locations to the GPU

In [ ]:
S64 = torch.exp(-2.9957 * torch.cdist(x, x) / 0.1)  # exponential covariance, effective range 0.1

In [ ]:
S32 = S64.float()  # 32-bit copy

In [ ]:
t = time.time(); L64 = torch.linalg.cholesky(S64); torch.cuda.synchronize(); time.time() - t  # double

In [ ]:
t = time.time(); L32 = torch.linalg.cholesky(S32); torch.cuda.synchronize(); time.time() - t  # single

In [ ]:
(L64 - L32).abs().max().item()  # largest entrywise difference

In [ ]:
2 * L64.diagonal().log().sum().item(), 2 * L32.diagonal().log().sum().item()  # log determinants

## Half and 8-bit precision on the GPU

Half precision: a quarter of the memory of double, and the matrix product runs on the tensor cores. The entries come out rounded. 8-bit floats store with an eighth of the memory, but there is no matrix product for them: storage only.

In [ ]:
S16 = S64.half()  # 16-bit copy

In [ ]:
S16.nbytes / 2**30  # memory in GB

In [ ]:
t = time.time(); P16 = S16 @ S16; torch.cuda.synchronize(); time.time() - t  # half

In [ ]:
t = time.time(); P64 = S64 @ S64; torch.cuda.synchronize(); time.time() - t  # double, for comparison

In [ ]:
P64[0, :3].tolist(), P16[0, :3].tolist()  # double, half

In [ ]:
((P16 - P64).abs() / P64).max().item()  # largest relative error

In [ ]:
S8 = S64.to(torch.float8_e4m3fn)  # 8-bit copy

In [ ]:
S8.nbytes / 2**30  # memory in GB

In [ ]:
(S8.double() - S64).abs().max().item()  # largest rounding error

In [ ]:
try:
    P8 = S8 @ S8  # no matrix product in 8-bit
except Exception as e:
    print(type(e).__name__, e)

## Reference

M. L. O. Salvaña, "Adaptive block mixed-precision Cholesky factorization for large-scale space-time Gaussian processes," in *Proc. SC26 Workshops of the International Conference for High Performance Computing, Networking, Storage, and Analysis (HPC4EES)*, Chicago, IL, USA, Nov. 2026.